# BLACK × Kaggriculture Research
公式環境をTruth sourceとして、ルール・Leaderboard・Replay・Opponent state・Market・Geometryを並列監査する。
原則: 一次情報 → 実装 → 実測 → 推測。Production policyはこのNotebookから直接変更しない。

In [ ]:
import json, os, subprocess, sys, statistics, math
from pathlib import Path
ART=Path('artifacts/kaggriculture_research'); ART.mkdir(parents=True, exist_ok=True)
print('research root:', ART.resolve())

## 1. Official environment contract
Kaggriculture is a 2-player, 720-turn farming simulation. The official environment exposes each player's farm state in `farms`; opponent private shed/inventory remains hidden. Market orders are ordered and capped at 10 per turn by default.

In [ ]:
from kaggle_environments import make
env = make('kaggriculture', configuration={'episodeSteps':720}, debug=False)
print('env ok')
print('agents:', ['pass','random','starter'])

## 2. Observation probe — opponent visibility
Non-destructive probe. Capture keys, farm structure, market/town fields, and verify that `farms[1-player]` is readable without inspecting private opponent inventory.

In [ ]:
def probe(seed=0):
    from kaggle_environments import make
    e=make('kaggriculture', configuration={'episodeSteps':1}, debug=False)
    e.run(['starter','starter'])
    obs=e.steps[0][0]['observation']
    print('obs keys:', sorted(obs.keys()))
    farms=obs.get('farms')
    print('farms type/len:', type(farms).__name__, len(farms) if farms is not None else None)
    if farms:
        for i,f in enumerate(farms): print('farm',i,'keys=',sorted(f.keys()))
probe()

## 3. Parallel replay/leaderboard collection
If Kaggle CLI credentials exist, collect leaderboard, own submissions, and episodes. These artifacts are evidence only; do not hard-code strategies from a single run.

In [ ]:
def sh(cmd):
    print('$',cmd)
    p=subprocess.run(cmd,shell=True,text=True,capture_output=True)
    print(p.stdout[:8000]); print(p.stderr[:2000]); return p.returncode
sh('kaggle competitions leaderboard kaggriculture -s -p 200 --csv > artifacts/kaggriculture_research/leaderboard.csv || true')
sh('kaggle competitions submissions kaggriculture --csv > artifacts/kaggriculture_research/submissions.csv || true')

## 4. Geometry / action-economy metrics
Measure Manhattan movement, work-turn share, market-order count, land expansion timing, hire timing, pickup/drop frequency, and terminal liquidation. Do not infer causality from correlation alone.

In [ ]:
def manhattan(a,b): return abs(a[0]-b[0])+abs(a[1]-b[1])
print('geometry scorer ready')

## 5. Opponent divergence
Compute only from public opponent farm state: crop mix, animal mix, unlocked land, hire count and public farm occupancy. Never use opponent private shed/inventory.
Primary hypothesis: opponent pressure should alter SELL ordering only; production/planting remains frozen.

In [ ]:
PRODUCTS=['WHEAT','CARROT','TOMATO','STRAWBERRY','MELON','FERTILIZER']
def opponent_signature(farm):
    return {
      'money':farm.get('money'),
      'hands':farm.get('farmHands', farm.get('hands')),
      'unlocked':farm.get('unlockedQuadrants', farm.get('unlocked_quadrants')),
      'tiles':len(farm.get('tiles',[])) if isinstance(farm.get('tiles'),list) else None,
    }
print('opponent signature ready')

## 6. Paired A/B gate
A = current Phase3 baseline.
B = baseline + sparse opponent-aware SELL adapter.
Promotion requires paired seeds, both seats, no runtime regressions, and a statistically defensible improvement. Cash and ladder rating are separate metrics.

In [ ]:
SEEDS=[42,1000,1050,1100,1200,1500,2026,300257]
print('paired seeds:', SEEDS)
print('GATE: never promote from one replay')

## 7. Evidence ledger
Record each result with source class: `OFFICIAL`, `IMPLEMENTATION`, `MEASURED`, `INFERENCE`. Any strategy promotion must cite a measured artifact.

In [ ]:
ledger=[]
def add_evidence(claim,source_class,evidence):
    ledger.append({'claim':claim,'source_class':source_class,'evidence':evidence})
add_evidence('Opponent farm is public in observation','OFFICIAL','farms[1-player] probe')
Path(ART/'evidence_ledger.json').write_text(json.dumps(ledger,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(ledger,ensure_ascii=False,indent=2))